In [59]:
import csv
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from pyproj import Transformer
import re
import scipy as sp
import pyreadr
import sys
sys.path.append(os.path.abspath('../src/'))
from visualize import *
from utils import *
from scipy import stats
import argparse
import copy
import glob
import argparse
import cProfile
import csv
from functools import partial
import itertools
import math
import matplotlib.pyplot as plt
import multiprocessing as mp
import numpy as np
import os
import rasterio
from scipy import stats
from scipy.optimize import minimize
# from scipy.special import binom
from skimage import graph
import sys
import time
from tqdm import tqdm
import numpy.ma as ma
import warnings
import numpy as np
import psutil
import gc

warnings.filterwarnings("ignore")

import logging
logging.getLogger("distributed").setLevel(logging.ERROR)

In [66]:
def compute_expected_n(ac_locs, trap_locs, g0, sigma, K, density, prob_cap, trap_x):
    """
    Computes expected number of unique individuals detected in a spatial capture-recapture study.
        ac_locs (2D array): Shape (num_activity_centers, 2) - x and y coordinates of activity centers.
        trap_locs (2D array): Shape (num_traps, 2) - x and y coordinates of all potential trap locations.
        g0 (float): Detection probability at the activity center.
        sigma (float): Scale parameter of the detection function.
        K (int): Number of sampling periods.
        density (float): Density of individuals per unit area.
        prob_cap (2D array): Shape (num_traps, num_activity_centers) - capture probability at each trap j of individuals with activity center l.
        trap_x (1D array): 1D array of activated trap locations.
    """
    for t in range(len(trap_locs)): # for trap locations that are not selected, set all capture probs to 0
        if int(trap_x[t]) == 0:
            prob_cap[t,...] = np.zeros(ac_locs.shape[0])

    i_cap_hist = np.squeeze(np.zeros((len(trap_locs), 1)))  # Initialize empty capture history 
    p_empty_cap_hist = compute_cond_lik_ind(len(ac_locs), prob_cap, K, len(trap_locs), i_cap_hist)
    p_nonempty = 1 - p_empty_cap_hist
    expected_n = np.sum(p_nonempty*density)
    return expected_n

def compute_cond_lik_ind(num_activity_centers, est_prob_cap, K, num_traps, ind_cap_hist):
    """
    Computes the likelihood of an individual's capture history conditional on their activity center location. 
        est_prob_cap (2D array): Shape (num_traps, num_activity_centers) - capture probability at each trap j of individuals with activity center l.   
        num_activity_centers (int): Number of potential activity centers. 
        K (int): Number of sampling periods.
        num_traps (int): Number of potential trap locations.
        ind_cap_hist (1D array): Capture history of an individual.
    """
    broadcast_i_cap_hist = np.broadcast_to(ind_cap_hist[:, np.newaxis], (num_traps, num_activity_centers))      # Reshapes to array of shaep (num_traps, num_activity_centers)
    probs = stats.binom.pmf(broadcast_i_cap_hist, K, est_prob_cap)
    zero_mask = probs == 0.0
    log_probs = np.log(probs, where=np.invert(zero_mask))
    log_probs[zero_mask] = -sys.maxsize - 1
    log_cond_lik_sums = np.sum(log_probs, axis=0)
    return np.exp(log_cond_lik_sums)

def compute_expected_n_across_scenarios(ac_locs, trap_locs, g0, sigma, K, density, prob_cap, trap_x):
    nscenarios = len(g0)
    e_n = np.zeros((nscenarios,1))
    for s in range(nscenarios):
        e_n[s,0] = compute_expected_n(ac_locs, trap_locs, g0[s], sigma[s], K, density[s], prob_cap[s], trap_x)
    return(e_n)

In [67]:
def compute_expected_c(ac_locs, trap_locs, g0, sigma, K, density, prob_cap, trap_x):
    """
    Computes expected number of captures.
        ac_locs (2D array): Shape (num_activity_centers, 2) - x and y coordinates of activity centers.
        trap_locs (2D array): Shape (num_traps, 2) - x and y coordinates of all potential trap locations.
        g0 (float): Detection probability at the activity center.
        sigma (float): Scale parameter of the detection function.
        K (int): Number of sampling periods.
        density (float): Density of individuals per unit area.
        prob_cap (2D array): Shape (num_traps, num_activity_centers) - capture probability at each trap j of individuals with activity center l.
        trap_x (1D array): 1D array of activated trap locations.
    """
    density_array = np.array(density).flatten()
    
    # For trap locations that are not selected, set all capture probs to 0
    for t in range(len(trap_locs)):
        if int(trap_x[t]) == 0:
            prob_cap[t,...] = np.zeros(ac_locs.shape[0])
    
    # Compute the expected number of captures
    broadcast_density = np.broadcast_to(density_array, (len(trap_locs), len(density_array)))    # Repeats density to shape of (num_traps, num_activity_centers)
    expected_c = np.sum(prob_cap*broadcast_density)*K
    return expected_c

def compute_expected_c_across_scenarios(ac_locs, trap_locs, g0, sigma, K, density, prob_cap, trap_x):
    nscenarios = len(g0)
    e_c = np.zeros((nscenarios,1))
    for s in range(nscenarios):
        e_c[s,0] = compute_expected_c(ac_locs, trap_locs, g0[s], sigma[s], K, density[s], prob_cap [s], trap_x)
    return(e_c)

In [68]:
def backward_greedy(scenarios, trap_loc, centers, K, distances):
    # Initailize all potential trap locations to have a camera.
    trap_x = np.ones((len(trap_loc),))

    # Initailize parameter storage across scenarios
    D = []
    g0 = []
    sigma = []
    density_prior = []
    alpha1 = []
    prob_cap = []

    # Intiailize storage of loss functions
    E_n_curr = []
    E_c_curr = []
    E_r_curr = []
    RSE_curr = []
    RSE_hist = []

    # For each scenario, calculate the RSE in the event all potential trap locations are activated.
    for s in range(len(scenarios)):
        D.append(scenarios[s][0])
        g0.append(scenarios[s][1])
        sigma.append(scenarios[s][2])
        density_prior.append(np.ones((centers.shape[0]))*(47/float(centers.shape[0])))      # List of Length # of potential activity centers
        # Read in density prior file from /data/density if not using uniform density
        # density_prior_file =  f'data/density/Dmod_draw_{s+1}.csv'
        # density_df = pd.read_csv(density_prior_file)
        # density_grid = density_df.pivot_table(values='D_mod',index='x',columns='y', fill_value = 0).astype(np.float64).values.T 
        # density_prior.append(density_grid)
        alpha1.append(1/(2*sigma[s]*sigma[s]))
        prob_cap.append((g0[s])*np.exp(-alpha1[s]*(distances**2)))      # Array of size (# traps, # poential activity centers)

        # Compute RSE for the current scenario when all trap locations are activated
        E_n_curr.append(compute_expected_n(centers, trap_loc, g0[s], sigma[s], K, density_prior[s], prob_cap[s], trap_x))
        E_c_curr.append(compute_expected_c(centers, trap_loc, g0[s], sigma[s], K, density_prior[s], prob_cap[s], trap_x))
        E_r_curr.append(E_c_curr[s] - E_n_curr[s])
        RSE_curr.append(1/np.sqrt(min([E_n_curr[s], E_r_curr[s]])))

    # Average performance across all scenarios when all trap locations are activated
    print("Computing Avg RSE Across Scenarios")
    RSE_hist.append(np.mean(RSE_curr))
    remove_hist = []

    # Begin backward greedy approach - removing 1 camera at a time
    counter = 0
    while sum(trap_x) > 60:    # Set mininum number of cameras -- dependent on scenario
        counter += 1           # Tracks number of camera removals
        print(counter)
        trap_indices = [i for i, x in enumerate(trap_x) if int(x) == 1]       # Locations where traps which are still activated
        trap_x_temp = np.copy(np.broadcast_to(trap_x, (len(trap_indices),trap_x.shape[0])))    # 2D array of shape (num_activated_trap_locs, num_possible_trap_locs)

        # Temporarily remove each currently active location to 0 and simulate the performance
        for pos in range(len(trap_indices)):
            trap_idx = trap_indices[pos]
            trap_x_temp[pos, trap_idx] = 0 

        # Multiprocess the E_n and E_c calculations
        func1 = partial(compute_expected_n_across_scenarios, centers, trap_loc, g0, sigma, K, density_prior, prob_cap)   # Setup all parameters but the activated trap locations
        func2 = partial(compute_expected_c_across_scenarios, centers, trap_loc, g0, sigma, K, density_prior, prob_cap)
        pool = mp.Pool(min(mp.cpu_count(), 10))
        E_n_per_scenario = np.squeeze(np.array(pool.map(func1, trap_x_temp)))
        E_c_per_scenario = np.squeeze(np.array(pool.map(func2, trap_x_temp)))
        pool.close()
        pool.join()

        # Calculate RSE
        E_r_per_scenario = E_c_per_scenario
        min_n_r_per_scenario = np.zeros(E_r_per_scenario.shape)
        RSE_per_scenario = np.zeros(E_r_per_scenario.shape)
        for t in range(len(trap_indices)):
            for s in range(len(scenarios)):
                E_r_per_scenario[t,s] -= E_n_per_scenario[t,s]
                min_n_r_per_scenario[t,s] = min(E_n_per_scenario[t,s], E_r_per_scenario[t,s])
                RSE_per_scenario[t,s] = 1/np.sqrt(min_n_r_per_scenario[t,s])
        RSE_temp = np.mean(RSE_per_scenario, axis=1).tolist()

        # Select which trap to remove based on RSE
        min_change_idx = RSE_temp.index(min(RSE_temp)) 
        remove = trap_indices[min_change_idx]
        trap_x[remove] = 0

        # Track removal history
        RSE_hist.append(RSE_temp[min_change_idx])
        remove_hist.append(remove)
        print(remove, RSE_temp[min_change_idx])
    
    return(remove_hist, RSE_hist)


In [69]:
# Read in parameter draws
params = pd.read_csv('data/params/New_sigma_range_2-27-25_param_values_for_each_draw300_2-27-25.csv')
params = params.rename(columns={'Unnamed: 0': 'index'})

# Extract parameter values
D = params['D'].values
g0 = params['g0'].values
sigma = params['sigma'].values
K= 5    # Number of sampling periods
# raster_cell_size = np.full(params.shape[0], 100) -- Not needed since using UTM coordinates

# Read in potential activity center locations
ac_coords = pyreadr.read_r('./data/grid/New_sigma_range_2-27-25_100m_mask_2-27-25.RDS')
ac_coords = ac_coords[None]
ac_coords_list = ac_coords[['x', 'y']].values.tolist()
ac_coords_list = np.array(ac_coords_list)

# Read in potential trap locations
trap_coords = pd.read_csv('./data/trap_locations/New_sigma_range_2-27-25_100m_trap_grid_2-27-25.csv')
trap_coords = trap_coords.drop(columns = ['Unnamed: 0'])
trap_coords = trap_coords.rename(columns = {'X': 'x', 'Y': 'y'})
trap_coords_list = []
for i in range(trap_coords.shape[0]):
    trap_coords_list.append((trap_coords['x'][i], trap_coords['y'][i]))
trap_coords_list = np.array(trap_coords_list)

# Randomly select 1000 trap locations and activity centers -- Comment out when not testing
np.random.seed(42)
np.random.shuffle(ac_coords_list)
ac_coords_list = (ac_coords_list)[:100]
np.random.shuffle(trap_coords_list)
trap_coords_list = (trap_coords_list)[:100]

# Calculate euclidean distances from traps to activity centers
traps = trap_coords_list[:, np.newaxis, :]  #(5, 1, 2)
centers = ac_coords_list[np.newaxis, :, :]  #(1, 5, 2)
differences = traps - centers
distances = np.linalg.norm(differences, axis=2)

scenarios = list(zip(*[D, g0, sigma]))

backward_greedy(scenarios, trap_coords_list, ac_coords_list, K, distances)

Computing Avg RSE Across Scenarios
1
81 0.17876499695025405
2
75 0.1787736988984363
3
86 0.17877726725948112
4
96 0.17879022657071408
5
76 0.17880617700952042
6
80 0.17881833981570153
7
82 0.17883482298495204
8
42 0.17885382125750726
9
43 0.17885836236350444
10
56 0.1788624328606787
11
53 0.17886762601523923
12
73 0.17887471080083958
13
15 0.17889464067466254
14
59 0.17890758874628035
15
19 0.17892644492483295
16
29 0.1789316026178812
17
20 0.17895098853989025
18
88 0.17897278702295202
19
31 0.17899830491571087
20
41 0.17900505087799626
21
2 0.17901873768328333
22
36 0.1790427371356991
23
65 0.17906855281177073
24
77 0.17909626894048863
25
7 0.17912460073493416
26
64 0.17916210238381003
27
51 0.17920183167374676
28
58 0.17924343439139925
29
30 0.17928664947835232
30
57 0.17933385084013337
31
12 0.17939027880013683
32
38 0.17944805907832656
33
22 0.1795171244825841
34
79 0.17957974600844256
35
74 0.17964676208945038
36
27 0.1797172432677372
37
23 0.17979014682412686
38
40 0.179883082038

([81,
  75,
  86,
  96,
  76,
  80,
  82,
  42,
  43,
  56,
  53,
  73,
  15,
  59,
  19,
  29,
  20,
  88,
  31,
  41,
  2,
  36,
  65,
  77,
  7,
  64,
  51,
  58,
  30,
  57,
  12,
  38,
  22,
  79,
  74,
  27,
  23,
  40,
  10,
  14],
 [0.17875745282844002,
  0.17876499695025405,
  0.1787736988984363,
  0.17877726725948112,
  0.17879022657071408,
  0.17880617700952042,
  0.17881833981570153,
  0.17883482298495204,
  0.17885382125750726,
  0.17885836236350444,
  0.1788624328606787,
  0.17886762601523923,
  0.17887471080083958,
  0.17889464067466254,
  0.17890758874628035,
  0.17892644492483295,
  0.1789316026178812,
  0.17895098853989025,
  0.17897278702295202,
  0.17899830491571087,
  0.17900505087799626,
  0.17901873768328333,
  0.1790427371356991,
  0.17906855281177073,
  0.17909626894048863,
  0.17912460073493416,
  0.17916210238381003,
  0.17920183167374676,
  0.17924343439139925,
  0.17928664947835232,
  0.17933385084013337,
  0.17939027880013683,
  0.17944805907832656,
  0.17